<a href="https://colab.research.google.com/github/kokami236/osiro1/blob/main/%E5%AD%A6%E7%BF%92%E3%83%87%E3%83%BC%E3%82%BF%E6%95%B4%E5%BD%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install open3d tqdm

import open3d as o3d
import numpy as np
import os, random, shutil
from collections import defaultdict
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.1 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [3]:
pip install open3d numpy

色変更

In [9]:
import open3d as o3d
import numpy as np

def remove_red_and_pink_points(input_file, output_file):
    # 1. 点群データの読み込み
    print(f"読み込み中: {input_file}")
    pcd = o3d.io.read_point_cloud(input_file)

    # 点群が空でないか確認
    if pcd.is_empty():
        print("エラー: 点群データが読み込めませんでした。")
        return

    # 2. 色情報の取得 (Open3Dでは色は 0.0 ~ 1.0 の範囲で扱われます)
    # colors は [N, 3] の配列で、各行が [R, G, B] を表します
    colors = np.asarray(pcd.colors)

    if len(colors) == 0:
        print("エラー: この点群には色情報が含まれていません。")
        return

    # 3. 「赤・濃い赤・ピンク」の条件定義
    # ---------------------------------------------------------
    # 【調整パラメータ】

    # (A) 赤色の最低ライン
    # 暗い赤（影のような赤）も消したいので、低めに設定します。
    # 0.15くらいあれば、真っ黒(0.0)な影以外の「暗い赤」を拾えます。
    r_min = 0.15

    # (B) 他の色との差分（ここが一番重要！）
    # 「赤色が、緑や青よりどれだけ強ければ消すか」という値。
    # ピンク色（R=0.9, G=0.6, B=0.6）などは差が小さいので、この値を小さくすると拾えます。
    # 目安: 0.1 〜 0.15 (小さくしすぎると茶色の木材まで消えるので注意)
    diff_threshold = 0.12

    # ---------------------------------------------------------

    # 条件判定ロジック：
    # 1. 赤成分がある程度以上ある (r_min)
    # 2. 赤成分が、緑成分よりも diff_threshold 分だけ強い
    # 3. 赤成分が、青成分よりも diff_threshold 分だけ強い

    red_indices = np.where(
        (colors[:, 0] > r_min) &
        (colors[:, 0] > colors[:, 1] + diff_threshold) &
        (colors[:, 0] > colors[:, 2] + diff_threshold)
    )[0]

    print(f"削除対象の点数: {len(red_indices)} / 全点数: {len(colors)}")
    print(f"削除率: {len(red_indices)/len(colors)*100:.2f}%")

    # 4. 赤い点以外のインデックスを取得して抽出
    # 全インデックスから赤い点のインデックスを除外
    pcd_filtered = pcd.select_by_index(red_indices, invert=True)

    # 5. 保存
    o3d.io.write_point_cloud(output_file, pcd_filtered)
    print(f"保存完了: {output_file}")

    # (オプション) 結果の表示（Google Colab等ではコメントアウト推奨）
    # o3d.visualization.draw_geometries([pcd_filtered], window_name="Filtered Result")

# --- 実行部分 ---
# ファイル名はご提示いただいたパスを使用しています
input_ply = "/content/drive/MyDrive/ジオラマ/掛川城推論前薄い.ply"
output_ply = "/content/drive/MyDrive/ジオラマ/掛川城推論前薄い_除去後.ply" # 分かりやすく名前を変えています

remove_red_and_pink_points(input_file=input_ply, output_file=output_ply)

読み込み中: /content/drive/MyDrive/ジオラマ/掛川城推論前薄い.ply
削除対象の点数: 70145 / 全点数: 387636
削除率: 18.10%
保存完了: /content/drive/MyDrive/ジオラマ/掛川城推論前薄い_除去後.ply


kesoon2の大きさの確認

In [ ]:
import numpy as np, open3d as o3d

def check_scale(ply_path):
    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points)
    mins, maxs = pts.min(0), pts.max(0)
    size = maxs - mins
    diag = np.linalg.norm(size)
    print("file:", ply_path)
    print("AABB size [x,y,z]:", size)
    print("diag:", diag)

check_scale("")


file: /content/drive/MyDrive/ジオラマ/kesson2.ply
AABB size [x,y,z]: [4.99355674 3.29972279 3.44459391]
diag: 6.90572274061408


正解データの確認（ここで学習用データをほかのスケールのものと合わせる。）今は推論用欠損データと正解データすでに同じ大きさだから、矢倉をそれと比較すればいい。

In [ ]:
import numpy as np, open3d as o3d

def diag(p):
    pts = np.asarray(o3d.io.read_point_cloud(p).points)
    size = pts.max(0) - pts.min(0)
    return float(np.linalg.norm(size)), size

a = "/content/drive/MyDrive/ジオラマ/kesson2.ply"
b = "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"

da, sa = diag(a)
db, sb = diag(b)

print("A size:", sa, "diag:", da)
print("B size:", sb, "diag:", db)
print("ratio A/B:", da/db)


A size: [4.99355674 3.29972279 3.44459391] diag: 6.90572274061408
B size: [4.99355674 3.29972279 3.44459391] diag: 6.90572274061408
ratio A/B: 1.0


512点以上で2視点あるもののみ採用

In [8]:
import os, sys, glob, math, random, itertools, json
import numpy as np
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
# 設定
# ==============================================================================
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply",
    "/content/drive/MyDrive/ジオラマ/小倉城out2.ply",
    "/content/drive/MyDrive/ジオラマ/udo-downsampled - Cloud.ply"
]

# “shapenet_pc” までの親
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset"

CATEGORY_ID = "02691156"
CATEGORY_NAME = "airplane"

PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048 # SeedFormer/PCN系の固定点数

# フィルタリング条件
MIN_POINTS_COMPLETE = PATCH_N
MIN_KEEP_VIEW = 256
MIN_VIEWS_PER_SAMPLE = 2

# 欠損生成
HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45

# splits
TRAIN_SPLIT = 0.85
VAL_SPLIT   = 0.10
# test は残り

RNG_SEED = 42
N_RENDERINGS = 5
APPEND_MODE = False # 基本はFalse（作り直し）
MAX_SAVE_POINTS = 4096

# ==============================================================================
# 出力先
# ==============================================================================
BASE = os.path.join(OUT_ROOT, "shapenet_pc", CATEGORY_ID, "train", "custom")
PARTIAL_ROOT  = os.path.join(BASE, "partial",  CATEGORY_NAME)
COMPLETE_ROOT = os.path.join(BASE, "complete", CATEGORY_NAME)
SPLIT_DIR     = os.path.join(BASE, "splits")
CUSTOM_JSON   = os.path.join(BASE, "Custom.json")

# ==============================================================================
# 初期化
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

# APPEND_MODEがFalseなら、フォルダを一度空にして ID=0 からスタートさせる
if (not APPEND_MODE) and os.path.exists(BASE):
    import shutil
    print("🧹 Removing:", BASE)
    shutil.rmtree(BASE, ignore_errors=True)

os.makedirs(PARTIAL_ROOT,  exist_ok=True)
os.makedirs(COMPLETE_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# ==============================================================================
# utility
# ==============================================================================
def sample_points_no_repeat(pts, n_points):
    if len(pts) < n_points:
        return None
    idx = np.random.choice(len(pts), n_points, replace=False)
    return pts[idx]

def pc_norm_unit_sphere(pts):
    ctr = np.mean(pts, axis=0)
    pts0 = pts - ctr
    scale = np.max(np.linalg.norm(pts0, axis=1)) + 1e-8
    return pts0 / scale

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    if diag <= 1e-12: return pts
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    if len(pts) == 0: return pts
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    if diameter <= 1e-12: return pts
    camera_dist = diameter * 100.0

    if view_axis == 'top': camera = [0, camera_dist, 0]
    elif view_axis == 'bottom': camera = [0, -camera_dist, 0]
    elif view_axis == 'front': camera = [0, 0, camera_dist]
    elif view_axis == 'side': camera = [camera_dist, 0, 0]
    else: camera = [0, 0, camera_dist]

    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000.0)
    if len(pt_map) == 0: return np.empty((0, 3), dtype=np.float32)
    return pts[np.asarray(pt_map)].astype(np.float32)

def maybe_clip(pts, max_points):
    if max_points is None: return pts
    if len(pts) <= max_points: return pts
    idx = np.random.choice(len(pts), max_points, replace=False)
    return pts[idx]

def write_pcd(path, pts):
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
    o3d.io.write_point_cloud(path, pcd, write_ascii=False)

patterns = [
    (0, lambda p: create_random_hole(p)),
    (1, lambda p: create_viewpoint_scan(p, 'top')),
    (2, lambda p: create_viewpoint_scan(p, 'front')),
    (3, lambda p: create_viewpoint_scan(p, 'side')),
    (4, lambda p: create_viewpoint_scan(p, 'bottom')),
]

# ==============================================================================
# 開始IDの設定
# ==============================================================================
# ループの外で global_id を初期化します。
# これにより、ファイルが変わってもカウンターはリセットされず、連番が続きます。
global_id = 0

# もし追記モードなら、既存の最大IDを探してそこから続きを振る
if APPEND_MODE:
    existing = sorted(glob.glob(os.path.join(COMPLETE_ROOT, "*.pcd")))
    if existing:
        try:
            ids = [int(os.path.splitext(os.path.basename(f))[0].split('-')[-1]) for f in existing]
            global_id = max(ids) + 1
        except Exception:
            pass

print("🎬 Start")
print("BASE:", BASE)
print("Start ID:", global_id)

all_model_ids = []

# ==============================================================================
# メイン生成
# ==============================================================================
for target_ply in TARGET_FILES:
    # ログファイル名用に安全な名前を作る（スペース対策）
    original_name = os.path.splitext(os.path.basename(target_ply))[0]
    safe_name = original_name.replace(" ", "_").replace("-", "_")

    print(f"\n🔄 Processing: {original_name} (log: {safe_name})")

    if not os.path.exists(target_ply):
        print("❌ Missing:", target_ply)
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points).astype(np.float32)
    if len(gt_pts) == 0:
        print("❌ Empty point cloud")
        continue

    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    log_file = os.path.join(BASE, f"log_{safe_name}.txt")
    processed = set()
    if APPEND_MODE and os.path.exists(log_file):
        with open(log_file, "r") as f:
            processed = set(line.strip() for line in f)

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(itertools.product(x_steps, y_steps, z_steps),
                            total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):

            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed:
                continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2], dtype=np.float32)
            minb   = np.array([x, y, z], dtype=np.float32)
            maxb   = minb + PATCH_SIZE

            k, idx, _ = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            cand = gt_pts[idx]
            mask = np.all((cand >= minb) & (cand < maxb), axis=1)
            patch = cand[mask]
            if len(patch) < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            patch_norm = pc_norm_unit_sphere(patch)
            comp_pts = sample_points_no_repeat(patch_norm, PATCH_N)
            if comp_pts is None:
                log_f.write(key + "\n")
                continue

            # ==========================================================
            # ID生成部分: 純粋な連番 (000000, 000001 ...)
            # ==========================================================
            model_id = f"{CATEGORY_ID}-{global_id:06d}"

            pdir = os.path.join(PARTIAL_ROOT, model_id)
            os.makedirs(pdir, exist_ok=True)

            saved = 0
            for vi, func in patterns:
                dense = func(patch_norm)
                if len(dense) < MIN_KEEP_VIEW:
                    continue
                dense = maybe_clip(dense, MAX_SAVE_POINTS)
                write_pcd(os.path.join(pdir, f"{vi:02d}.pcd"), dense)
                saved += 1

            if saved < MIN_VIEWS_PER_SAMPLE:
                import shutil
                shutil.rmtree(pdir, ignore_errors=True)
                log_f.write(key + "\n")
                continue

            write_pcd(os.path.join(COMPLETE_ROOT, f"{model_id}.pcd"), comp_pts)

            all_model_ids.append(model_id)

            # ここでインクリメントすることで、次のパッチ、次のファイルのときも
            # 番号が途切れずに続きます
            global_id += 1

            log_f.write(key + "\n")

# ==============================================================================
# splits と Custom.json
# ==============================================================================
print("\n📝 Writing splits / Custom.json ...")

if len(all_model_ids) == 0:
    print("⚠️ No samples generated.")
    raise SystemExit(0)

# IDリストをシャッフルして分割
random.shuffle(all_model_ids)
n = len(all_model_ids)

n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_ids = all_model_ids[:n_train]
val_ids   = all_model_ids[n_train:n_train + n_val]
test_ids  = all_model_ids[n_train + n_val:]

with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
    f.write("\n".join(train_ids))
with open(os.path.join(SPLIT_DIR, "val.txt"), "w") as f:
    f.write("\n".join(val_ids))
with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
    f.write("\n".join(test_ids))

custom_index = [{
    "taxonomy_id": CATEGORY_ID,
    "taxonomy_name": CATEGORY_NAME,
    "train": train_ids,
    "val": val_ids,
    "test": test_ids
}]
with open(CUSTOM_JSON, "w") as f:
    json.dump(custom_index, f, indent=2)

print("✅ Done")
print(" COMPLETE:", COMPLETE_ROOT)
print(" PARTIAL :", PARTIAL_ROOT)
print(" SPLITS  :", SPLIT_DIR)
print(" Custom.json:", CUSTOM_JSON)
print(" total:", n, "train:", len(train_ids), "val:", len(val_ids), "test:", len(test_ids))
print(" settings: MIN_KEEP_VIEW=", MIN_KEEP_VIEW,
      "MIN_VIEWS_PER_SAMPLE=", MIN_VIEWS_PER_SAMPLE,
      "MAX_SAVE_POINTS=", MAX_SAVE_POINTS)

# ==============================================================================
# おまけ：生成後の簡易統計（view数分布）
# ==============================================================================
from collections import Counter

view_hist = Counter()
for mid in all_model_ids:
    pdir = os.path.join(PARTIAL_ROOT, mid)
    cnt = 0
    for i in range(N_RENDERINGS):
        if os.path.exists(os.path.join(pdir, f"{i:02d}.pcd")):
            cnt += 1
    view_hist[cnt] += 1

print("\n=== view count distribution (0..4 exist) ===")
for k in sorted(view_hist.keys()):
    print(f"{k} views:", view_hist[k])

🎬 Start
BASE: /content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset/shapenet_pc/02691156/train/custom
Start ID: 0

🔄 Processing: 熊本正解データ (log: 熊本正解データ)


Slicing 熊本正解データ: 100%|██████████| 57750/57750 [03:26<00:00, 279.91it/s]



🔄 Processing: 小倉城out2 (log: 小倉城out2)


Slicing 小倉城out2: 100%|██████████| 20412/20412 [03:36<00:00, 94.27it/s] 



🔄 Processing: udo-downsampled - Cloud (log: udo_downsampled___Cloud)


Slicing udo_downsampled___Cloud: 100%|██████████| 54243/54243 [01:35<00:00, 565.92it/s] 



📝 Writing splits / Custom.json ...
✅ Done
 COMPLETE: /content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset/shapenet_pc/02691156/train/custom/complete/airplane
 PARTIAL : /content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset/shapenet_pc/02691156/train/custom/partial/airplane
 SPLITS  : /content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset/shapenet_pc/02691156/train/custom/splits
 Custom.json: /content/drive/MyDrive/My_PCN_Dataset260/My_PCN_Dataset/shapenet_pc/02691156/train/custom/Custom.json
 total: 7011 train: 5959 val: 701 test: 351
 settings: MIN_KEEP_VIEW= 256 MIN_VIEWS_PER_SAMPLE= 2 MAX_SAVE_POINTS= 4096

=== view count distribution (0..4 exist) ===
2 views: 736
3 views: 1393
4 views: 2961
5 views: 1921


In [ ]:
import os, sys, glob, math, random, itertools, json
import numpy as np
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
# 設定
# ==============================================================================
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply",
    "/content/drive/MyDrive/ジオラマ/小倉城out2.ply",
    "/content/drive/MyDrive/ジオラマ/udo-downsampled - Cloud.ply"
]

# “shapenet_pc” までの親（この下に shapenet_pc/02691156/train/custom/... を作る）
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset"

CATEGORY_ID = "02691156"
CATEGORY_NAME = "airplane"

PATCH_SIZE = 0.25
STRIDE     = 0.1

# SeedFormer/PCN系の固定点数
PATCH_N = 2048

# complete候補条件（この点数未満のパッチは捨てる）
MIN_POINTS_COMPLETE = PATCH_N

# partial view の採用条件（dataloaderの min_keep と合わせる）
MIN_KEEP_VIEW = 256

# Top-k=2 を成立させる（最低 view 数）
MIN_VIEWS_PER_SAMPLE = 2

# 欠損生成（ランダムhole）
HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45

# splits
TRAIN_SPLIT = 0.85   # 小規模なら 0.85/0.10/0.05 くらいが安定
VAL_SPLIT   = 0.10
# test は残り

RNG_SEED = 42

# view数（00..04 を “作れた分だけ” 保存）
N_RENDERINGS = 5

# 既存データに追記するなら True（基本は作り直し推奨）
APPEND_MODE = False

# partial保存が重い場合の上限（Noneなら無制限）
MAX_SAVE_POINTS = 4096   # 例: 4096 / 8192 / None

# ==============================================================================
# 出力先（cfg.DATASETS.CUSTOM.* に合わせる）
# ==============================================================================
BASE = os.path.join(OUT_ROOT, "shapenet_pc", CATEGORY_ID, "train", "custom")
PARTIAL_ROOT  = os.path.join(BASE, "partial",  CATEGORY_NAME)   # .../<model_id>/<vi>.pcd
COMPLETE_ROOT = os.path.join(BASE, "complete", CATEGORY_NAME)   # .../<model_id>.pcd
SPLIT_DIR     = os.path.join(BASE, "splits")
CUSTOM_JSON   = os.path.join(BASE, "Custom.json")

# ==============================================================================
# 初期化
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

if (not APPEND_MODE) and os.path.exists(BASE):
    import shutil
    print("🧹 Removing:", BASE)
    shutil.rmtree(BASE, ignore_errors=True)

os.makedirs(PARTIAL_ROOT,  exist_ok=True)
os.makedirs(COMPLETE_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# ==============================================================================
# utility
# ==============================================================================
def sample_points_no_repeat(pts, n_points):
    """不足ならNone（replace=True禁止）"""
    if len(pts) < n_points:
        return None
    idx = np.random.choice(len(pts), n_points, replace=False)
    return pts[idx]

def pc_norm_unit_sphere(pts):
    """中心合わせ＋unit sphere正規化"""
    ctr = np.mean(pts, axis=0)
    pts0 = pts - ctr
    scale = np.max(np.linalg.norm(pts0, axis=1)) + 1e-8
    return pts0 / scale

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    if diag <= 1e-12:
        return pts
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    """HPRで '見える点' を返す（可変長）"""
    if len(pts) == 0:
        return pts

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)

    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    if diameter <= 1e-12:
        return pts

    camera_dist = diameter * 100.0
    if view_axis == 'top':
        camera = [0, camera_dist, 0]
    elif view_axis == 'bottom':
        camera = [0, -camera_dist, 0]
    elif view_axis == 'front':
        camera = [0, 0, camera_dist]
    elif view_axis == 'side':
        camera = [camera_dist, 0, 0]
    else:
        camera = [0, 0, camera_dist]

    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000.0)
    if len(pt_map) == 0:
        return np.empty((0, 3), dtype=np.float32)
    return pts[np.asarray(pt_map)].astype(np.float32)

def maybe_clip(pts, max_points):
    if max_points is None:
        return pts
    if len(pts) <= max_points:
        return pts
    idx = np.random.choice(len(pts), max_points, replace=False)
    return pts[idx]

def write_pcd(path, pts):
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
    o3d.io.write_point_cloud(path, pcd, write_ascii=False)

patterns = [
    (0, lambda p: create_random_hole(p)),
    (1, lambda p: create_viewpoint_scan(p, 'top')),
    (2, lambda p: create_viewpoint_scan(p, 'front')),
    (3, lambda p: create_viewpoint_scan(p, 'side')),
    (4, lambda p: create_viewpoint_scan(p, 'bottom')),
]

# ==============================================================================
# 開始ID
# ==============================================================================
existing = sorted(glob.glob(os.path.join(COMPLETE_ROOT, "*.pcd")))
global_id = 0
if existing:
    try:
        ids = [int(os.path.splitext(os.path.basename(f))[0].split('-')[-1]) for f in existing]
        global_id = max(ids) + 1
    except Exception:
        pass

print("🎬 Start")
print("BASE:", BASE)
print("Start ID:", global_id)

all_model_ids = []

# ==============================================================================
# メイン生成
# ==============================================================================
for target_ply in TARGET_FILES:
    safe_name = os.path.splitext(os.path.basename(target_ply))[0]
    print(f"\n🔄 Processing: {safe_name}")

    if not os.path.exists(target_ply):
        print("❌ Missing:", target_ply)
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points).astype(np.float32)
    if len(gt_pts) == 0:
        print("❌ Empty point cloud")
        continue

    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    # AABB内の点を確実に拾える半径（中心から半対角）
    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    log_file = os.path.join(BASE, f"log_{safe_name}.txt")
    processed = set()
    if APPEND_MODE and os.path.exists(log_file):
        with open(log_file, "r") as f:
            processed = set(line.strip() for line in f)

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(itertools.product(x_steps, y_steps, z_steps),
                            total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):

            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed:
                continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2], dtype=np.float32)
            minb   = np.array([x, y, z], dtype=np.float32)
            maxb   = minb + PATCH_SIZE

            k, idx, _ = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            cand = gt_pts[idx]
            mask = np.all((cand >= minb) & (cand < maxb), axis=1)
            patch = cand[mask]
            if len(patch) < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            # 正規化（中心＋unit sphere）
            patch_norm = pc_norm_unit_sphere(patch)

            # complete（重複なしで2048抽出）
            comp_pts = sample_points_no_repeat(patch_norm, PATCH_N)
            if comp_pts is None:
                log_f.write(key + "\n")
                continue

            # model_id = f"{CATEGORY_ID}-{global_id:06d}"
            # safe_name（例: 熊本正解データ, udo-downsampled...）をIDに含める
            # ファイル名が重複しないようになります
            model_id = f"{CATEGORY_ID}-{safe_name}-{global_id:06d}"
            pdir = os.path.join(PARTIAL_ROOT, model_id)
            os.makedirs(pdir, exist_ok=True)

            # partial view：可変長で保存（min_keep以上）
            saved = 0
            for vi, func in patterns:
                dense = func(patch_norm)
                if len(dense) < MIN_KEEP_VIEW:
                    continue

                dense = maybe_clip(dense, MAX_SAVE_POINTS)
                write_pcd(os.path.join(pdir, f"{vi:02d}.pcd"), dense)
                saved += 1

            # Top-k(=2)が成立しないなら破棄
            if saved < MIN_VIEWS_PER_SAMPLE:
                import shutil
                shutil.rmtree(pdir, ignore_errors=True)
                log_f.write(key + "\n")
                continue

            # complete保存
            write_pcd(os.path.join(COMPLETE_ROOT, f"{model_id}.pcd"), comp_pts)

            all_model_ids.append(model_id)
            global_id += 1
            log_f.write(key + "\n")

# ==============================================================================
# splits と Custom.json
# ==============================================================================
print("\n📝 Writing splits / Custom.json ...")

if len(all_model_ids) == 0:
    print("⚠️ No samples generated.")
    raise SystemExit(0)

random.shuffle(all_model_ids)
n = len(all_model_ids)

n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_ids = all_model_ids[:n_train]
val_ids   = all_model_ids[n_train:n_train + n_val]
test_ids  = all_model_ids[n_train + n_val:]

with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
    f.write("\n".join(train_ids))
with open(os.path.join(SPLIT_DIR, "val.txt"), "w") as f:
    f.write("\n".join(val_ids))
with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
    f.write("\n".join(test_ids))

custom_index = [{
    "taxonomy_id": CATEGORY_ID,
    "taxonomy_name": CATEGORY_NAME,
    "train": train_ids,
    "val": val_ids,
    "test": test_ids
}]
with open(CUSTOM_JSON, "w") as f:
    json.dump(custom_index, f, indent=2)

print("✅ Done")
print(" COMPLETE:", COMPLETE_ROOT)
print(" PARTIAL :", PARTIAL_ROOT)
print(" SPLITS  :", SPLIT_DIR)
print(" Custom.json:", CUSTOM_JSON)
print(" total:", n, "train:", len(train_ids), "val:", len(val_ids), "test:", len(test_ids))
print(" settings: MIN_KEEP_VIEW=", MIN_KEEP_VIEW,
      "MIN_VIEWS_PER_SAMPLE=", MIN_VIEWS_PER_SAMPLE,
      "MAX_SAVE_POINTS=", MAX_SAVE_POINTS)

# ==============================================================================
# おまけ：生成後の簡易統計（view数分布）
# ==============================================================================
from collections import Counter

view_hist = Counter()
for mid in all_model_ids:
    pdir = os.path.join(PARTIAL_ROOT, mid)
    cnt = 0
    for i in range(N_RENDERINGS):
        if os.path.exists(os.path.join(pdir, f"{i:02d}.pcd")):
            cnt += 1
    view_hist[cnt] += 1

print("\n=== view count distribution (0..4 exist) ===")
for k in sorted(view_hist.keys()):
    print(f"{k} views:", view_hist[k])


🎬 Start
BASE: /content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset/shapenet_pc/02691156/train/custom
Start ID: 0

🔄 Processing: 熊本正解データ


Slicing 熊本正解データ: 100%|██████████| 57750/57750 [03:25<00:00, 280.46it/s]



🔄 Processing: 小倉城out2


Slicing 小倉城out2: 100%|██████████| 20412/20412 [03:39<00:00, 93.04it/s] 



🔄 Processing: udo-downsampled - Cloud


Slicing udo-downsampled - Cloud: 100%|██████████| 54243/54243 [01:38<00:00, 549.58it/s] 



📝 Writing splits / Custom.json ...
✅ Done
 COMPLETE: /content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset/shapenet_pc/02691156/train/custom/complete/airplane
 PARTIAL : /content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset/shapenet_pc/02691156/train/custom/partial/airplane
 SPLITS  : /content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset/shapenet_pc/02691156/train/custom/splits
 Custom.json: /content/drive/MyDrive/My_PCN_Dataset259/My_PCN_Dataset/shapenet_pc/02691156/train/custom/Custom.json
 total: 7011 train: 5959 val: 701 test: 351
 settings: MIN_KEEP_VIEW= 256 MIN_VIEWS_PER_SAMPLE= 2 MAX_SAVE_POINTS= 4096

=== view count distribution (0..4 exist) ===
2 views: 736
3 views: 1393
4 views: 2961
5 views: 1921


個々から先は使わない

In [ ]:
import os, sys, glob, math, random, itertools, json
import numpy as np
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
# 設定
# ==============================================================================
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"
]

OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset"
CATEGORY_ID = "02691156"
CATEGORY_NAME = "airplane"

PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048

# completeは重複なしで2048取れることが条件
MIN_POINTS_COMPLETE = PATCH_N

# view選別（dataloaderのmin_keepと揃えるのがコツ）
MIN_KEEP_VIEW = 256          # ★ここを min_keep に合わせる
MIN_VIEWS_PER_SAMPLE = 2     # ★Top-k(=2)が成立する最小view数

HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45

TRAIN_SPLIT = 0.90
VAL_SPLIT   = 0.05           # ★valを空にしない
RNG_SEED    = 42
N_RENDERINGS = 5             # 00..04 を「作れたぶんだけ」保存

APPEND_MODE = False

# ==============================================================================
# 出力先
# ==============================================================================
BASE = os.path.join(OUT_ROOT, "shapenet_pc", CATEGORY_ID, "train", "custom")
PARTIAL_ROOT  = os.path.join(BASE, "partial",  CATEGORY_NAME)
COMPLETE_ROOT = os.path.join(BASE, "complete", CATEGORY_NAME)
SPLIT_DIR     = os.path.join(BASE, "splits")
CUSTOM_JSON   = os.path.join(BASE, "Custom.json")

# ==============================================================================
# 初期化
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

if (not APPEND_MODE) and os.path.exists(BASE):
    import shutil
    print("🧹 Removing:", BASE)
    shutil.rmtree(BASE, ignore_errors=True)

os.makedirs(PARTIAL_ROOT,  exist_ok=True)
os.makedirs(COMPLETE_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# ==============================================================================
# 関数
# ==============================================================================
def sample_points_no_repeat(pts, n_points):
    """不足ならNone（replace=True禁止）"""
    if len(pts) < n_points:
        return None
    idx = np.random.choice(len(pts), n_points, replace=False)
    return pts[idx]

def pc_norm_unit_sphere(pts):
    """中心合わせ＋unit sphere正規化（ShapeNet系に近い）"""
    ctr = np.mean(pts, axis=0)
    pts0 = pts - ctr
    scale = np.max(np.linalg.norm(pts0, axis=1)) + 1e-8
    return pts0 / scale

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    if diag <= 1e-12:
        return pts
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    if len(pts) == 0:
        return pts
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)

    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    if diameter <= 1e-12:
        return pts

    camera_dist = diameter * 100.0
    if view_axis == 'top':
        camera = [0, camera_dist, 0]
    elif view_axis == 'bottom':
        camera = [0, -camera_dist, 0]
    elif view_axis == 'front':
        camera = [0, 0, camera_dist]
    elif view_axis == 'side':
        camera = [camera_dist, 0, 0]
    else:
        camera = [0, 0, camera_dist]

    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000.0)
    if len(pt_map) == 0:
        return np.empty((0, 3), dtype=np.float32)
    return pts[np.asarray(pt_map)]

def write_pcd(path, pts):
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
    o3d.io.write_point_cloud(path, pcd, write_ascii=False)

patterns = [
    (0, lambda p: create_random_hole(p)),
    (1, lambda p: create_viewpoint_scan(p, 'top')),
    (2, lambda p: create_viewpoint_scan(p, 'front')),
    (3, lambda p: create_viewpoint_scan(p, 'side')),
    (4, lambda p: create_viewpoint_scan(p, 'bottom')),
]

# ==============================================================================
# 開始ID
# ==============================================================================
existing = sorted(glob.glob(os.path.join(COMPLETE_ROOT, "*.pcd")))
global_id = 0
if existing:
    try:
        ids = [int(os.path.splitext(os.path.basename(f))[0].split('-')[-1]) for f in existing]
        global_id = max(ids) + 1
    except Exception:
        pass

print("🎬 Start")
print("BASE:", BASE)
print("Start ID:", global_id)

all_model_ids = []

# ==============================================================================
# メイン生成
# ==============================================================================
for target_ply in TARGET_FILES:
    safe_name = os.path.splitext(os.path.basename(target_ply))[0]
    print(f"\n🔄 Processing: {safe_name}")

    if not os.path.exists(target_ply):
        print("❌ Missing:", target_ply)
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points).astype(np.float32)
    if len(gt_pts) == 0:
        print("❌ Empty point cloud")
        continue

    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    log_file = os.path.join(BASE, f"log_{safe_name}.txt")
    processed = set()
    if APPEND_MODE and os.path.exists(log_file):
        with open(log_file, "r") as f:
            processed = set(line.strip() for line in f)

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(itertools.product(x_steps, y_steps, z_steps),
                            total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):

            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed:
                continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2], dtype=np.float32)
            minb = np.array([x, y, z], dtype=np.float32)
            maxb = minb + PATCH_SIZE

            k, idx, _ = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            cand = gt_pts[idx]
            mask = np.all((cand >= minb) & (cand < maxb), axis=1)
            patch = cand[mask]
            if len(patch) < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n")
                continue

            # 正規化（中心＋unit sphere）
            patch_norm = pc_norm_unit_sphere(patch)

            # complete（重複なし）
            comp_pts = sample_points_no_repeat(patch_norm, PATCH_N)
            if comp_pts is None:
                log_f.write(key + "\n")
                continue

            model_id = f"{CATEGORY_ID}-{global_id:06d}"
            pdir = os.path.join(PARTIAL_ROOT, model_id)
            os.makedirs(pdir, exist_ok=True)

            # 5 view生成：薄いviewは保存しない。view数が2未満なら捨てる
            saved = 0
            for vi, func in patterns:
                dense = func(patch_norm)

                # viewが薄いならスキップ（採用率のため）
                if len(dense) < MIN_KEEP_VIEW:
                    continue

                # 2048点が「重複なし」で取れる view だけ保存
                part_pts = sample_points_no_repeat(dense, PATCH_N)
                if part_pts is None:
                    continue

                write_pcd(os.path.join(pdir, f"{vi:02d}.pcd"), part_pts)
                saved += 1

            # Top-k(=2)が成立しないなら破棄
            if saved < MIN_VIEWS_PER_SAMPLE:
                import shutil
                shutil.rmtree(pdir, ignore_errors=True)
                log_f.write(key + "\n")
                continue

            # complete保存
            write_pcd(os.path.join(COMPLETE_ROOT, f"{model_id}.pcd"), comp_pts)

            all_model_ids.append(model_id)
            global_id += 1
            log_f.write(key + "\n")

# ==============================================================================
# splits と Custom.json
# ==============================================================================
print("\n📝 Writing splits / Custom.json ...")

if len(all_model_ids) == 0:
    print("⚠️ No samples generated.")
    raise SystemExit(0)

random.shuffle(all_model_ids)
n = len(all_model_ids)

n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_ids = all_model_ids[:n_train]
val_ids   = all_model_ids[n_train:n_train + n_val]
test_ids  = all_model_ids[n_train + n_val:]

with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
    f.write("\n".join(train_ids))
with open(os.path.join(SPLIT_DIR, "val.txt"), "w") as f:
    f.write("\n".join(val_ids))
with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
    f.write("\n".join(test_ids))

custom_index = [{
    "taxonomy_id": CATEGORY_ID,
    "taxonomy_name": CATEGORY_NAME,
    "train": train_ids,
    "val": val_ids,
    "test": test_ids
}]
with open(CUSTOM_JSON, "w") as f:
    json.dump(custom_index, f, indent=2)

print("✅ Done")
print(" COMPLETE:", COMPLETE_ROOT)
print(" PARTIAL :", PARTIAL_ROOT)
print(" SPLITS  :", SPLIT_DIR)
print(" Custom.json:", CUSTOM_JSON)
print(" total:", n, "train:", len(train_ids), "val:", len(val_ids), "test:", len(test_ids))
print(" settings: MIN_KEEP_VIEW=", MIN_KEEP_VIEW, "MIN_VIEWS_PER_SAMPLE=", MIN_VIEWS_PER_SAMPLE)


🎬 Start
BASE: /content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset/shapenet_pc/02691156/train/custom
Start ID: 0

🔄 Processing: 熊本正解データ


Slicing 熊本正解データ: 100%|██████████| 57750/57750 [02:15<00:00, 425.68it/s]


📝 Writing splits / Custom.json ...
✅ Done
 COMPLETE: /content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset/shapenet_pc/02691156/train/custom/complete/airplane
 PARTIAL : /content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset/shapenet_pc/02691156/train/custom/partial/airplane
 SPLITS  : /content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset/shapenet_pc/02691156/train/custom/splits
 Custom.json: /content/drive/MyDrive/My_PCN_Dataset256/My_PCN_Dataset/shapenet_pc/02691156/train/custom/Custom.json
 total: 47 train: 42 val: 2 test: 3
 settings: MIN_KEEP_VIEW= 256 MIN_VIEWS_PER_SAMPLE= 2


In [ ]:
import os, glob, numpy as np
import open3d as o3d

COMPLETE_DIR = "/content/drive/MyDrive/My_PCN_Dataset2/PCN/train/complete/complete"
PARTIAL_DIR  = "/content/drive/MyDrive/My_PCN_Dataset2/PCN/train/partial/partial"

def load_pcd(path):
    pcd = o3d.io.read_point_cloud(path)
    return np.asarray(pcd.points)

def unique_ratio(pts):
    if len(pts) == 0:
        return 0.0
    u = np.unique(np.round(pts, 6), axis=0)
    return len(u) / len(pts)

# complete を1つ取る
complete_files = sorted(glob.glob(os.path.join(COMPLETE_DIR, "*.pcd")))
print("complete files:", len(complete_files))
cpath = complete_files[0]
stem = os.path.splitext(os.path.basename(cpath))[0]  # 02691156-000000

comp = load_pcd(cpath)
print("COMP:", stem, "N=", len(comp), "unique_ratio=", unique_ratio(comp))

# partial はフォルダ名 = stem で並んでる想定
pdir = os.path.join(PARTIAL_DIR, stem)
print("partial dir:", pdir, "exists=", os.path.isdir(pdir))

for suf in ["00","01","02","03","04"]:
    p = os.path.join(pdir, f"{suf}.pcd")
    if not os.path.exists(p):
        print(" -", suf, "MISSING")
        continue
    pts = load_pcd(p)
    print(" -", suf, "N=", len(pts), "unique_ratio=", unique_ratio(pts))


complete files: 6930
COMP: 02691156-000000 N= 2048 unique_ratio= 0.03759765625
partial dir: /content/drive/MyDrive/My_PCN_Dataset2/PCN/train/partial/partial/02691156-000000 exists= True
 - 00 N= 2048 unique_ratio= 0.02880859375
 - 01 N= 2048 unique_ratio= 0.02783203125
 - 02 N= 2048 unique_ratio= 0.013671875
 - 03 N= 2048 unique_ratio= 0.01611328125
 - 04 N= 2048 unique_ratio= 0.02783203125


In [ ]:
import os, sys, glob, math, random, itertools, json
import numpy as np
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
# 設定
# ==============================================================================
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"
]

# ここは “shapenet_pc” までの親を指定（この下に shapenet_pc/02691156/train/custom/... を作る）
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset"
CATEGORY_ID = "02691156"
CATEGORY_NAME = "airplane"

PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048

# ★水増し禁止：complete作成に必要な最低点数
MIN_POINTS_COMPLETE = PATCH_N

# view成立の最小可視点（少なすぎるviewは捨てる）
MIN_VISIBLE_RATIO = 0.35
MIN_POINTS_VIEW   = int(PATCH_N * MIN_VISIBLE_RATIO)

HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45

TRAIN_SPLIT = 0.9
RNG_SEED    = 42
N_RENDERINGS = 5  # 00..04 を作る

# 壊れたデータと混ぜないなら False 推奨（作り直し）
APPEND_MODE = False

# ==============================================================================
# 出力先（あなたの cfg の CUSTOM パスに一致）
# ==============================================================================
BASE = os.path.join(OUT_ROOT, "shapenet_pc", CATEGORY_ID, "train", "custom")
PARTIAL_ROOT  = os.path.join(BASE, "partial",  CATEGORY_NAME)   # .../%s/%02d.pcd
COMPLETE_ROOT = os.path.join(BASE, "complete", CATEGORY_NAME)   # .../%s.pcd
SPLIT_DIR     = os.path.join(BASE, "splits")
CUSTOM_JSON   = os.path.join(BASE, "Custom.json")               # cfg.DATASETS.CUSTOM.CATEGORY_FILE_PATH 用

# ==============================================================================
# 初期化
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

if (not APPEND_MODE) and os.path.exists(BASE):
    import shutil
    print("🧹 Removing:", BASE)
    shutil.rmtree(BASE, ignore_errors=True)

os.makedirs(PARTIAL_ROOT,  exist_ok=True)
os.makedirs(COMPLETE_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# ==============================================================================
# 関数
# ==============================================================================
def sample_points_no_repeat(pts, n_points):
    """不足ならNone（replace=True禁止）"""
    if len(pts) < n_points:
        return None
    idx = np.random.choice(len(pts), n_points, replace=False)
    return pts[idx]

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    if len(pts) == 0:
        return pts
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    if diameter <= 1e-12:
        return pts
    camera_dist = diameter * 100.0
    if view_axis == 'top':
        camera = [0, camera_dist, 0]
    elif view_axis == 'bottom':
        camera = [0, -camera_dist, 0]
    elif view_axis == 'front':
        camera = [0, 0, camera_dist]
    elif view_axis == 'side':
        camera = [camera_dist, 0, 0]
    else:
        camera = [0, 0, camera_dist]
    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000.0)
    if len(pt_map) == 0:
        return np.empty((0, 3), dtype=np.float32)
    return pts[np.asarray(pt_map)]

def write_pcd(path, pts):
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
    o3d.io.write_point_cloud(path, pcd, write_ascii=False)

patterns = [
    (0, lambda p: create_random_hole(p)),
    (1, lambda p: create_viewpoint_scan(p, 'top')),
    (2, lambda p: create_viewpoint_scan(p, 'front')),
    (3, lambda p: create_viewpoint_scan(p, 'side')),
    (4, lambda p: create_viewpoint_scan(p, 'bottom')),
]

# ==============================================================================
# 開始ID
# ==============================================================================
existing = sorted(glob.glob(os.path.join(COMPLETE_ROOT, "*.pcd")))
global_id = 0
if existing:
    try:
        ids = [int(os.path.splitext(os.path.basename(f))[0].split('-')[-1]) for f in existing]
        global_id = max(ids) + 1
    except:
        pass

print("🎬 Start")
print("BASE:", BASE)
print("Start ID:", global_id)

all_model_ids = []  # train/test splits & Custom.json 用

# ==============================================================================
# メイン生成
# ==============================================================================
for target_ply in TARGET_FILES:
    safe_name = os.path.splitext(os.path.basename(target_ply))[0]
    print(f"\n🔄 Processing: {safe_name}")

    if not os.path.exists(target_ply):
        print("❌ Missing:", target_ply)
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points).astype(np.float32)
    if len(gt_pts) == 0:
        print("❌ Empty point cloud")
        continue

    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    log_file = os.path.join(BASE, f"log_{safe_name}.txt")
    processed = set()
    if APPEND_MODE and os.path.exists(log_file):
        with open(log_file, "r") as f:
            processed = set(line.strip() for line in f)

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(itertools.product(x_steps, y_steps, z_steps),
                            total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):

            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed:
                continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2], dtype=np.float32)
            minb = np.array([x, y, z], dtype=np.float32)
            maxb = minb + PATCH_SIZE

            k, idx, _ = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n"); continue

            cand = gt_pts[idx]
            mask = np.all((cand >= minb) & (cand < maxb), axis=1)
            patch = cand[mask]
            if len(patch) < MIN_POINTS_COMPLETE:
                log_f.write(key + "\n"); continue

            # 正規化（patch全体平均）
            ctr = np.mean(patch, axis=0)
            patch_norm = patch - ctr

            # complete（重複なし）
            comp_pts = sample_points_no_repeat(patch_norm, PATCH_N)
            if comp_pts is None:
                log_f.write(key + "\n"); continue

            model_id = f"{CATEGORY_ID}-{global_id:06d}"

            # partial出力先フォルダ（.../partial/airplane/<model_id>/）
            pdir = os.path.join(PARTIAL_ROOT, model_id)
            os.makedirs(pdir, exist_ok=True)

            # 5 view 生成：成立しないviewはスキップ（代用しない）
            saved = 0
            for vi, func in patterns:
                dense = func(patch_norm)
                if len(dense) < MIN_POINTS_VIEW:
                    continue
                part_pts = sample_points_no_repeat(dense, PATCH_N)
                if part_pts is None:
                    continue
                write_pcd(os.path.join(pdir, f"{vi:02d}.pcd"), part_pts)
                saved += 1

            # 1つもviewが作れないなら破棄
            if saved == 0:
                import shutil
                shutil.rmtree(pdir, ignore_errors=True)
                log_f.write(key + "\n"); continue

            # complete保存（.../complete/airplane/<model_id>.pcd）
            write_pcd(os.path.join(COMPLETE_ROOT, f"{model_id}.pcd"), comp_pts)

            all_model_ids.append(model_id)
            global_id += 1
            log_f.write(key + "\n")

# ==============================================================================
# splits と Custom.json を作る
# ==============================================================================
print("\n📝 Writing splits / Custom.json ...")

if len(all_model_ids) == 0:
    print("⚠️ No samples generated.")
    raise SystemExit(0)

random.shuffle(all_model_ids)
split = int(len(all_model_ids) * TRAIN_SPLIT)
train_ids = all_model_ids[:split]
test_ids  = all_model_ids[split:]

with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
    f.write("\n".join(train_ids))
with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
    f.write("\n".join(test_ids))

# CustomDataLoader が読む JSON（taxonomy_id と train/test の model_id リスト）
custom_index = [{
    "taxonomy_id": CATEGORY_ID,
    "taxonomy_name": CATEGORY_NAME,
    "train": train_ids,
    "val": [],
    "test": test_ids
}]
with open(CUSTOM_JSON, "w") as f:
    json.dump(custom_index, f, indent=2)

print("✅ Done")
print(" COMPLETE:", COMPLETE_ROOT)
print(" PARTIAL :", PARTIAL_ROOT)
print(" SPLITS  :", SPLIT_DIR)
print(" Custom.json:", CUSTOM_JSON)
print(" total:", len(all_model_ids), "train:", len(train_ids), "test:", len(test_ids))


🎬 Start
BASE: /content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom
Start ID: 0

🔄 Processing: 熊本正解データ


Slicing 熊本正解データ: 100%|██████████| 57750/57750 [02:18<00:00, 417.17it/s] 


📝 Writing splits / Custom.json ...
✅ Done
 COMPLETE: /content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom/complete/airplane
 PARTIAL : /content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom/partial/airplane
 SPLITS  : /content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom/splits
 Custom.json: /content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom/Custom.json
 total: 1168 train: 1051 test: 117


In [ ]:
import os, glob
BASE="/content/drive/MyDrive/My_PCN_Dataset4/My_PCN_Dataset/shapenet_pc/02691156/train/custom/partial/airplane"
ids=[d for d in os.listdir(BASE) if os.path.isdir(os.path.join(BASE,d))]
miss=0
for mid in ids:
    for i in range(5):
        if not os.path.exists(os.path.join(BASE, mid, f"{i:02d}.pcd")):
            miss += 1
            break
print("partial dirs:", len(ids), " missing_any_view:", miss)


partial dirs: 1168  missing_any_view: 1168


In [ ]:
import os, math, itertools
import numpy as np
from tqdm import tqdm
import open3d as o3d

GT_PLY = "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"

PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048

# まず「complete候補」として見る最低点数（ここはPATCH_NでOK）
MIN_PATCH_POINTS = PATCH_N

def create_viewpoint_scan_count(pts, view_axis):
    if len(pts) == 0:
        return 0
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)

    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    if diameter <= 1e-12:
        return len(pts)

    camera_dist = diameter * 100.0
    if view_axis == 'top':
        camera = [0, camera_dist, 0]
    elif view_axis == 'bottom':
        camera = [0, -camera_dist, 0]
    elif view_axis == 'front':
        camera = [0, 0, camera_dist]
    elif view_axis == 'side':
        camera = [camera_dist, 0, 0]
    else:
        camera = [0, 0, camera_dist]

    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000.0)
    return len(pt_map)

def summarize(arr, name):
    arr = np.asarray(arr)
    if len(arr) == 0:
        print(f"{name}: empty")
        return
    qs = [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]
    vals = np.percentile(arr, qs)
    print(f"\n{name} (N={len(arr)})")
    for q, v in zip(qs, vals):
        print(f"  p{q:>3}: {v:.0f}")
    print(f"  min={arr.min():.0f}  max={arr.max():.0f}")

gt_pcd = o3d.io.read_point_cloud(GT_PLY)
gt_pts = np.asarray(gt_pcd.points).astype(np.float32)
tree = o3d.geometry.KDTreeFlann(gt_pcd)

mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
x_steps = np.arange(mins[0], maxs[0], STRIDE)
y_steps = np.arange(mins[1], maxs[1], STRIDE)
z_steps = np.arange(mins[2], maxs[2], STRIDE)
total_steps = len(x_steps) * len(y_steps) * len(z_steps)

search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

patch_counts = []
top_counts   = []
front_counts = []
side_counts  = []
bottom_counts= []

valid_cells = 0

for x, y, z in tqdm(itertools.product(x_steps, y_steps, z_steps),
                    total=total_steps, mininterval=1.0, desc="Counting"):
    center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2], dtype=np.float32)
    minb   = np.array([x, y, z], dtype=np.float32)
    maxb   = minb + PATCH_SIZE

    k, idx, _ = tree.search_radius_vector_3d(center, search_radius)
    if k < MIN_PATCH_POINTS:
        continue

    cand = gt_pts[idx]
    mask = np.all((cand >= minb) & (cand < maxb), axis=1)
    patch = cand[mask]
    if len(patch) < MIN_PATCH_POINTS:
        continue

    valid_cells += 1
    patch_counts.append(len(patch))

    # 正規化（平均引き）
    patch_norm = patch - np.mean(patch, axis=0)

    # 各viewの「見える点数」
    top_counts.append(create_viewpoint_scan_count(patch_norm, 'top'))
    front_counts.append(create_viewpoint_scan_count(patch_norm, 'front'))
    side_counts.append(create_viewpoint_scan_count(patch_norm, 'side'))
    bottom_counts.append(create_viewpoint_scan_count(patch_norm, 'bottom'))

print("\nvalid cells (patch>=2048):", valid_cells)

summarize(patch_counts,  "patch raw count (AABB)")
summarize(top_counts,    "visible count: top")
summarize(front_counts,  "visible count: front")
summarize(side_counts,   "visible count: side")
summarize(bottom_counts, "visible count: bottom")

# 「全viewが2048以上」の割合も見る
all2048 = np.mean((np.array(top_counts)   >= PATCH_N) &
                  (np.array(front_counts) >= PATCH_N) &
                  (np.array(side_counts)  >= PATCH_N) &
                  (np.array(bottom_counts)>= PATCH_N))
print("\nratio(all 4 views >= 2048):", all2048)


Counting: 100%|██████████| 57750/57750 [01:44<00:00, 553.80it/s]


valid cells (patch>=2048): 2725

patch raw count (AABB) (N=2725)
  p  0: 2052
  p  1: 2075
  p  5: 2210
  p 10: 2383
  p 25: 2922
  p 50: 4018
  p 75: 5648
  p 90: 7473
  p 95: 8277
  p 99: 9698
  p100: 11361
  min=2052  max=11361

visible count: top (N=2725)
  p  0: 54
  p  1: 88
  p  5: 127
  p 10: 166
  p 25: 325
  p 50: 644
  p 75: 1034
  p 90: 1402
  p 95: 1656
  p 99: 2087
  p100: 2728
  min=54  max=2728

visible count: front (N=2725)
  p  0: 69
  p  1: 114
  p  5: 151
  p 10: 188
  p 25: 271
  p 50: 455
  p 75: 785
  p 90: 1187
  p 95: 1501
  p 99: 2063
  p100: 2383
  min=69  max=2383

visible count: side (N=2725)
  p  0: 58
  p  1: 103
  p  5: 134
  p 10: 175
  p 25: 265
  p 50: 422
  p 75: 622
  p 90: 874
  p 95: 1056
  p 99: 1531
  p100: 1903
  min=58  max=1903

visible count: bottom (N=2725)
  p  0: 67
  p  1: 99
  p  5: 133
  p 10: 157
  p 25: 239
  p 50: 425
  p 75: 746
  p 90: 1034
  p 95: 1226
  p 99: 1725
  p100: 2293
  min=67  max=2293

ratio(all 4 views >= 2048): 0.0

In [ ]:
import numpy as np

# 例: すでにあるやつを np.array にする
top    = np.asarray(top_counts,    dtype=np.float32)
front  = np.asarray(front_counts,  dtype=np.float32)
side   = np.asarray(side_counts,   dtype=np.float32)
bottom = np.asarray(bottom_counts, dtype=np.float32)

assert len(top)==len(front)==len(side)==len(bottom)
N = len(top)
print("N =", N)

X = np.stack([top, front, side, bottom], axis=1)
names = ["top", "front", "side", "bottom"]

# -----------------------------
# Pearson 相関
# -----------------------------
pearson = np.corrcoef(X, rowvar=False)

print("\n[Pearson correlation]")
print("        " + " ".join([f"{n:>7s}" for n in names]))
for i, n in enumerate(names):
    row = " ".join([f"{pearson[i,j]:7.3f}" for j in range(len(names))])
    print(f"{n:>7s} {row}")

# -----------------------------
# Spearman 相関（scipy無し版）
#  - 値を順位に変換して Pearson を取る
# -----------------------------
def rankdata(a):
    # ties は平均順位（簡易版）
    a = np.asarray(a)
    order = np.argsort(a, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(a), dtype=np.float64)

    # tie処理
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j+1 < len(a) and sorted_a[j+1] == sorted_a[i]:
            j += 1
        if j > i:
            avg = (i + j) / 2.0
            ranks[order[i:j+1]] = avg
        i = j + 1
    return ranks

R = np.stack([rankdata(top), rankdata(front), rankdata(side), rankdata(bottom)], axis=1)
spearman = np.corrcoef(R, rowvar=False)

print("\n[Spearman correlation]")
print("        " + " ".join([f"{n:>7s}" for n in names]))
for i, n in enumerate(names):
    row = " ".join([f"{spearman[i,j]:7.3f}" for j in range(len(names))])
    print(f"{n:>7s} {row}")

# -----------------------------
# “全方向密”の割合: min4 >= T
# -----------------------------
min4 = np.min(X, axis=1)

print("\n[min over 4 views] percentiles")
for p in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"  p{p:>3}: {np.percentile(min4, p):.0f}")

for T in [256, 512, 1024, 2048]:
    cnt = int(np.sum(min4 >= T))
    print(f"\nmin4 >= {T}: {cnt}/{N}  ({cnt/N*100:.2f}%)")

# -----------------------------
# “全方向密だけ採用”したら何個残る？
#  - ここで採用基準を好きなTにする
# -----------------------------
T = 512  # 例: 512以上を全方向密とみなす
keep_idx = np.where(min4 >= T)[0]
print(f"\n[KEEP] threshold T={T}: keep {len(keep_idx)} / {N}")

# 参考: keepしたときの各view中央値
if len(keep_idx) > 0:
    Xk = X[keep_idx]
    print("\n[KEEP stats] median counts")
    for i, n in enumerate(names):
        print(f"  {n:>6s}: {np.median(Xk[:,i]):.0f}")


N = 6595

[Pearson correlation]
            top   front    side  bottom
    top   1.000  -0.101  -0.169   0.553
  front  -0.101   1.000   0.420   0.055
   side  -0.169   0.420   1.000  -0.076
 bottom   0.553   0.055  -0.076   1.000

[Spearman correlation]
            top   front    side  bottom
    top   1.000  -0.117  -0.172   0.478
  front  -0.117   1.000   0.499   0.044
   side  -0.172   0.499   1.000  -0.053
 bottom   0.478   0.044  -0.053   1.000

[min over 4 views] percentiles
  p  0: 34
  p  1: 69
  p  5: 97
  p 10: 119
  p 25: 169
  p 50: 251
  p 75: 376
  p 90: 502
  p 95: 592
  p 99: 798
  p100: 1168

min4 >= 256: 3213/6595  (48.72%)

min4 >= 512: 609/6595  (9.23%)

min4 >= 1024: 5/6595  (0.08%)

min4 >= 2048: 0/6595  (0.00%)

[KEEP] threshold T=512: keep 609 / 6595

[KEEP stats] median counts
     top: 1011
   front: 1041
    side: 747
  bottom: 877


ここに意図的に欠損させるコード入れたい

正解データさえ入れれば意図的に欠損、上、下、横から欠損させて、どこからの情報でもきれいなお城を出力するようなデータセットに揃える完成バンコード（水増ししてたからアウト）


In [ ]:
import os
import sys
import glob
import numpy as np
import random
import itertools
import math
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    print("Open3Dがインストールされていません。インストールします...")
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
#  🛠️ 設定エリア (ここに追加したいファイルだけを書く)
# ==============================================================================

# ★ここには「新しく追加したいファイル（小倉城）」だけを書きます
# ※ 熊本城もやり直したい場合はここに熊本城のパスも書いて、OUT_ROOTの中身を空にしてください
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"
]

# ★保存先（前回と同じ場所）
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset2/PCN"
CATEGORY_ID = "02691156"

# パラメータ
PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048
MIN_POINTS = 50
HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45
TRAIN_SPLIT = 0.9
RNG_SEED    = 42

# ==============================================================================
#  初期化 & 関数定義
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

BASE_DIR = os.path.join(OUT_ROOT, "train")
COMPLETE_DIR = os.path.join(BASE_DIR, "complete", CATEGORY_ID)
PARTIAL_DIR  = os.path.join(BASE_DIR, "partial", CATEGORY_ID)
SPLIT_DIR    = os.path.join(OUT_ROOT, "splits")

os.makedirs(COMPLETE_DIR, exist_ok=True)
os.makedirs(PARTIAL_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

def sample_points(pts, n_points):
    if len(pts) == 0: return pts
    if len(pts) >= n_points:
        return pts[np.random.choice(len(pts), n_points, replace=False)]
    else:
        return pts[np.random.choice(len(pts), n_points, replace=True)]

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    camera_dist = diameter * 100
    if view_axis == 'top': camera = [0, camera_dist, 0]
    elif view_axis == 'bottom': camera = [0, -camera_dist, 0]
    elif view_axis == 'front': camera = [0, 0, camera_dist]
    elif view_axis == 'side': camera = [camera_dist, 0, 0]
    else: camera = [0, 0, camera_dist]
    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000)
    return pts[pt_map]

# ==============================================================================
#  🚀 メインループ
# ==============================================================================

# 既存ファイルのID確認
existing = glob.glob(os.path.join(COMPLETE_DIR, "*.pcd"))
global_patch_id = 0
if existing:
    try:
        ids = [int(os.path.basename(f).split('-')[-1].replace('.pcd','')) for f in existing]
        global_patch_id = max(ids) + 1
    except: pass

print(f"🎬 追加モードで開始します。")
print(f"   現在の最終ID: {global_patch_id - 1 if global_patch_id > 0 else 'なし'}")
print(f"   次の開始ID  : {global_patch_id}")

for target_ply in TARGET_FILES:
    safe_name = os.path.splitext(os.path.basename(target_ply))[0]
    print(f"\n🔄 処理中: {safe_name}")

    if not os.path.exists(target_ply):
        print(f"❌ ファイルが見つかりません: {target_ply}")
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points)
    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    log_file = f"{OUT_ROOT}/log_{safe_name}.txt"
    processed_keys = set()
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            for line in f: processed_keys.add(line.strip())

    grid_coords = itertools.product(x_steps, y_steps, z_steps)
    file_new_count = 0
    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(grid_coords, total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):
            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed_keys: continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2])

            # ★ここを修正しました (Numpy配列として定義)
            min_bound = np.array([x, y, z])
            max_bound = min_bound + PATCH_SIZE

            [k, idx, _] = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS:
                log_f.write(key + "\n"); continue

            cand = gt_pts[idx]

            # ★修正箇所: Numpy配列同士で比較
            mask = np.all((cand >= min_bound) & (cand < max_bound), axis=1)
            patch = cand[mask]

            if len(patch) < MIN_POINTS:
                log_f.write(key + "\n"); continue

            # Complete作成
            comp_pts = sample_points(patch, PATCH_N)
            comp_pts = comp_pts - np.mean(comp_pts, axis=0)

            # 保存
            file_stem = f"{CATEGORY_ID}-{global_patch_id:06d}"

            patterns = [
                ("00", lambda p: create_random_hole(p)),
                ("01", lambda p: create_viewpoint_scan(p, 'top')),
                ("02", lambda p: create_viewpoint_scan(p, 'front')),
                ("03", lambda p: create_viewpoint_scan(p, 'side')),
                ("04", lambda p: create_viewpoint_scan(p, 'bottom'))
            ]

            partial_subdir = os.path.join(PARTIAL_DIR, file_stem)
            os.makedirs(partial_subdir, exist_ok=True)
            valid_exists = False

            for suffix, func in patterns:
                part_pts = func(comp_pts)
                if len(part_pts) < MIN_POINTS: part_pts = create_random_hole(comp_pts)
                part_pts = sample_points(part_pts, PATCH_N)
                o3d.io.write_point_cloud(os.path.join(partial_subdir, f"{suffix}.pcd"), o3d.geometry.PointCloud(o3d.utility.Vector3dVector(part_pts)), write_ascii=False)
                valid_exists = True

            if valid_exists:
                o3d.io.write_point_cloud(os.path.join(COMPLETE_DIR, f"{file_stem}.pcd"), o3d.geometry.PointCloud(o3d.utility.Vector3dVector(comp_pts)), write_ascii=False)
                global_patch_id += 1
                file_new_count += 1

            log_f.write(key + "\n")

    print(f"   ✅ {safe_name} 追加完了: {file_new_count} 個")

# ==============================================================================
#  📝 リスト更新
# ==============================================================================
print("📝 Updating split lists...")
all_names = sorted([d for d in os.listdir(PARTIAL_DIR) if os.path.isdir(os.path.join(PARTIAL_DIR, d))])

if len(all_names) > 0:
    random.shuffle(all_names)
    split = int(len(all_names) * TRAIN_SPLIT)

    with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
        f.write("\n".join(all_names[:split]))
    with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
        f.write("\n".join(all_names[split:]))

    print(f"✅ 全データ統合リスト更新完了: 合計 {len(all_names)} 件")
else:
    print("⚠️ データがありません")

🎬 追加モードで開始します。
   現在の最終ID: なし
   次の開始ID  : 0

🔄 処理中: 熊本正解データ


Slicing 熊本正解データ: 100%|██████████| 57750/57750 [05:59<00:00, 160.60it/s]


   ✅ 熊本正解データ 追加完了: 6930 個
📝 Updating split lists...
✅ 全データ統合リスト更新完了: 合計 6930 件


さいしん　ファイル形式変更


In [ ]:
ls /content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom | wc -l
ls /content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom/complete/airplane | wc -l
ls /content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom/partial/airplane | wc -l


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (ipython-input-3142652739.py, line 1)

In [ ]:
import os
import shutil
import subprocess

# ================== 設定 ==================
SRC_ROOT   = "/content/drive/MyDrive/My_PCN_Dataset2/PCN"                  # 入力（PCN生成先）
DST_ROOT   = "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc"          # 出力先（shapenet_pc）
CATEGORY_ID = "02691156"
MODE = "copy"   # "copy" or "move"

# 速度優先なら tar.gz 推奨（zipより速いことが多い）
ARCHIVE_KIND = "tar"   # "tar" or "zip"
ARCHIVE_OUT  = f"/content/drive/MyDrive/My_PCN_Dataset3_custom_{CATEGORY_ID}.tar.gz"  # or .zip
# ===========================================

src_complete = os.path.join(SRC_ROOT, "train", "complete", CATEGORY_ID)
src_partial  = os.path.join(SRC_ROOT, "train", "partial",  CATEGORY_ID)
src_splits   = os.path.join(SRC_ROOT, "splits")

dst_base     = os.path.join(DST_ROOT, CATEGORY_ID, "train", "custom")
dst_complete = os.path.join(dst_base, "complete")
dst_partial  = os.path.join(dst_base, "partial")
dst_splits   = os.path.join(dst_base, "splits")

os.makedirs(dst_complete, exist_ok=True)
os.makedirs(dst_partial, exist_ok=True)
os.makedirs(dst_splits, exist_ok=True)

def do_copy(src, dst):
    if MODE == "move":
        shutil.move(src, dst)
    else:
        shutil.copy2(src, dst)

def do_copytree(src_dir, dst_dir):
    # src_dir の中身を dst_dir に再帰コピー/移動
    for root, _, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        out = os.path.join(dst_dir, rel) if rel != "." else dst_dir
        os.makedirs(out, exist_ok=True)
        for fn in files:
            s = os.path.join(root, fn)
            d = os.path.join(out, fn)
            if MODE == "move":
                shutil.move(s, d)
            else:
                shutil.copy2(s, d)

# ---------- 1) copy/move ----------
if not os.path.isdir(src_complete):
    raise FileNotFoundError(f"src_complete not found: {src_complete}")
if not os.path.isdir(src_partial):
    raise FileNotFoundError(f"src_partial not found: {src_partial}")

complete_files = [f for f in os.listdir(src_complete) if f.endswith(".pcd")]
print("complete files:", len(complete_files))
for fn in complete_files:
    do_copy(os.path.join(src_complete, fn), os.path.join(dst_complete, fn))

subdirs = [d for d in os.listdir(src_partial) if os.path.isdir(os.path.join(src_partial, d))]
print("partial dirs:", len(subdirs))
for d in subdirs:
    do_copytree(os.path.join(src_partial, d), os.path.join(dst_partial, d))

for name in ["train.txt", "test.txt"]:
    p = os.path.join(src_splits, name)
    if os.path.exists(p):
        do_copy(p, os.path.join(dst_splits, name))

print("DONE copy/move ->", dst_base, "MODE:", MODE)

# ---------- 2) add airplane folder inside DST ----------
for sub in ["complete", "partial"]:
    src = os.path.join(dst_base, sub)
    dst = os.path.join(dst_base, sub, "airplane")
    os.makedirs(dst, exist_ok=True)

    for name in os.listdir(src):
        if name == "airplane":
            continue
        shutil.move(os.path.join(src, name), os.path.join(dst, name))

print("DONE structure -> complete/airplane & partial/airplane")

# ---------- 3) archive (zip or tar.gz) ----------
# Drive上で大量ファイルを扱うより「圧縮して1ファイル」にするとUI反映やDLが楽
work_parent = os.path.dirname(dst_base)  # .../train
custom_dir_name = os.path.basename(dst_base)  # "custom"

if ARCHIVE_KIND == "zip":
    # zipは「cdして相対パス」でやると安定
    cmd = f'cd "{work_parent}" && zip -r "{ARCHIVE_OUT}" "{custom_dir_name}"'
else:
    # tar.gz は速いことが多い
    cmd = f'cd "{work_parent}" && tar -czf "{ARCHIVE_OUT}" "{custom_dir_name}"'

print("ARCHIVE CMD:", cmd)
subprocess.check_call(cmd, shell=True)

print("ARCHIVE OK ->", ARCHIVE_OUT)


complete files: 6930
partial dirs: 6930
DONE copy/move -> /content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom MODE: copy
DONE structure -> complete/airplane & partial/airplane
ARCHIVE CMD: cd "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train" && tar -czf "/content/drive/MyDrive/My_PCN_Dataset3_custom_02691156.tar.gz" "custom"


CalledProcessError: Command 'cd "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train" && tar -czf "/content/drive/MyDrive/My_PCN_Dataset3_custom_02691156.tar.gz" "custom"' returned non-zero exit status 1.

In [ ]:
import os
import shutil

# ====== ここだけ設定 ======
SRC_ROOT = "/content/drive/MyDrive/My_PCN_Dataset2/PCN"  # いまのPCN生成先
DST_ROOT = "/content/drive/MyDrive/My_PCN_Dataset4/shapenet_pc"  # 変換後の親
CATEGORY_ID = "02691156"

MODE = "copy"   # "copy" か "move"（まずはcopy推奨）
# =========================

src_complete = os.path.join(SRC_ROOT, "train", "complete", CATEGORY_ID)
src_partial  = os.path.join(SRC_ROOT, "train", "partial",  CATEGORY_ID)
src_splits   = os.path.join(SRC_ROOT, "splits")

dst_base     = os.path.join(DST_ROOT, CATEGORY_ID, "train", "custom")
dst_complete = os.path.join(dst_base, "complete")
dst_partial  = os.path.join(dst_base, "partial")
dst_splits   = os.path.join(dst_base, "splits")

os.makedirs(dst_complete, exist_ok=True)
os.makedirs(dst_partial, exist_ok=True)
os.makedirs(dst_splits, exist_ok=True)

def do_copy(src, dst):
    if MODE == "move":
        shutil.move(src, dst)
    else:
        shutil.copy2(src, dst)

def do_copytree(src_dir, dst_dir):
    if MODE == "move":
        # moveはディレクトリ単位で移すならOKだけど、既存があると面倒なので中身単位で
        for root, _, files in os.walk(src_dir):
            rel = os.path.relpath(root, src_dir)
            out = os.path.join(dst_dir, rel) if rel != "." else dst_dir
            os.makedirs(out, exist_ok=True)
            for fn in files:
                shutil.move(os.path.join(root, fn), os.path.join(out, fn))
    else:
        for root, _, files in os.walk(src_dir):
            rel = os.path.relpath(root, src_dir)
            out = os.path.join(dst_dir, rel) if rel != "." else dst_dir
            os.makedirs(out, exist_ok=True)
            for fn in files:
                shutil.copy2(os.path.join(root, fn), os.path.join(out, fn))

# 1) complete/*.pcd
if not os.path.isdir(src_complete):
    raise FileNotFoundError(f"src_complete not found: {src_complete}")
complete_files = [f for f in os.listdir(src_complete) if f.endswith(".pcd")]
print("complete files:", len(complete_files))
for fn in complete_files:
    do_copy(os.path.join(src_complete, fn), os.path.join(dst_complete, fn))

# 2) partial/<model_id>/*.pcd
if not os.path.isdir(src_partial):
    raise FileNotFoundError(f"src_partial not found: {src_partial}")
subdirs = [d for d in os.listdir(src_partial) if os.path.isdir(os.path.join(src_partial, d))]
print("partial dirs:", len(subdirs))
for d in subdirs:
    do_copytree(os.path.join(src_partial, d), os.path.join(dst_partial, d))

# 3) splits を移す（train.txt, test.txt があれば）
for name in ["train.txt", "test.txt"]:
    p = os.path.join(src_splits, name)
    if os.path.exists(p):
        do_copy(p, os.path.join(dst_splits, name))

print("DONE ->", dst_base)
print("MODE:", MODE)


complete files: 6930
partial dirs: 6930
DONE -> /content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom
MODE: copy


In [ ]:
import os, shutil

BASE = "/content/drive/MyDrive/My_PCN_Dataset2/shapenet_pc/02691156/train/custom"

for sub in ["complete", "partial"]:
    src = os.path.join(BASE, sub)
    dst = os.path.join(BASE, sub, "airplane")
    os.makedirs(dst, exist_ok=True)

    for name in os.listdir(src):
        if name == "airplane":
            continue
        shutil.move(os.path.join(src, name), os.path.join(dst, name))

print("OK: airplane フォルダを追加して配置を揃えました")



OK: airplane フォルダを追加して配置を揃えました


In [ ]:
zip -r /content/drive/MyDrive/custom.zip \
    /content/drive/MyDrive/My_PCN_Dataset2/shapenet_pc/02691156/train/custom
	zip warning: name not matched: /content/drive/MyDrive/My_PCN_Dataset2/shapenet_pc/02691156/train/custom

zip error: Nothing to do! (try: zip -r /content/drive/MyDrive/custom.zip . -i /content/drive/MyDrive/My_PCN_Dataset2/shapenet_pc/02691156/train/custom)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (ipython-input-505938339.py, line 2)

In [ ]:
!ls "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom/complete/airplane" | head
!ls "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom/partial/airplane" | head
!ls "/content/drive/MyDrive/My_PCN_Dataset3/shapenet_pc/02691156/train/custom/partial/airplane/02691156-000000"



02691156-000000.pcd
02691156-000001.pcd
02691156-000002.pcd
02691156-000003.pcd
02691156-000004.pcd
02691156-000005.pcd
02691156-000006.pcd
02691156-000007.pcd
02691156-000008.pcd
02691156-000009.pcd
02691156-000000
02691156-000001
02691156-000002
02691156-000003
02691156-000004
02691156-000005
02691156-000006
02691156-000007
02691156-000008
02691156-000009
00.pcd	01.pcd	02691156-000000  02.pcd  03.pcd  04.pcd


In [ ]:
import os
import sys
import glob
import numpy as np
import random
import itertools
import math
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    print("Open3Dがインストールされていません。インストールします...")
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
#  🛠️ 設定エリア (ここに追加したいファイルだけを書く)
# ==============================================================================

# ★ここには「新しく追加したいファイル（小倉城）」だけを書きます
TARGET_FILES = [
    "/content/drive/MyDrive/ジオラマ/熊本正解データ.ply"
]

# ★保存先は「熊本城」を作ったときと【全く同じ場所】を指定してください
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset/PCN"
CATEGORY_ID = "02691156"

# パラメータ（前回と同じにする）
PATCH_SIZE = 0.25
STRIDE     = 0.1
PATCH_N    = 2048
MIN_POINTS = 50
HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45
TRAIN_SPLIT = 0.9
RNG_SEED    = 42

# ==============================================================================
#  初期化 & 関数定義
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

BASE_DIR = os.path.join(OUT_ROOT, "train")
COMPLETE_DIR = os.path.join(BASE_DIR, "complete", CATEGORY_ID)
PARTIAL_DIR  = os.path.join(BASE_DIR, "partial", CATEGORY_ID)
SPLIT_DIR    = os.path.join(OUT_ROOT, "splits")

os.makedirs(COMPLETE_DIR, exist_ok=True)
os.makedirs(PARTIAL_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

def sample_points(pts, n_points):
    if len(pts) == 0: return pts
    if len(pts) >= n_points:
        return pts[np.random.choice(len(pts), n_points, replace=False)]
    else:
        return pts[np.random.choice(len(pts), n_points, replace=True)]

def create_random_hole(pts):
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)
    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius]

def create_viewpoint_scan(pts, view_axis):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    camera_dist = diameter * 100
    if view_axis == 'top': camera = [0, camera_dist, 0]
    elif view_axis == 'bottom': camera = [0, -camera_dist, 0]
    elif view_axis == 'front': camera = [0, 0, camera_dist]
    elif view_axis == 'side': camera = [camera_dist, 0, 0]
    else: camera = [0, 0, camera_dist]
    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000)
    return pts[pt_map]

# ==============================================================================
#  🚀 メインループ
# ==============================================================================

# ★ここがポイント: フォルダ内の既存ファイルを見て、最後の番号の「次」を取得します
existing = glob.glob(os.path.join(COMPLETE_DIR, "*.pcd"))
global_patch_id = 0
if existing:
    try:
        # ファイル名 (例: 02691156-000500.pcd) から数字を取り出して最大値を探す
        ids = [int(os.path.basename(f).split('-')[-1].replace('.pcd','')) for f in existing]
        global_patch_id = max(ids) + 1
    except: pass

print(f"🎬 追加モードで開始します。")
print(f"   現在の最終ID: {global_patch_id - 1}")
print(f"   次の開始ID  : {global_patch_id} (ここから小倉城を追加します)")

for target_ply in TARGET_FILES:
    safe_name = os.path.splitext(os.path.basename(target_ply))[0]
    print(f"\n🔄 処理中: {safe_name}")

    if not os.path.exists(target_ply):
        print(f"❌ ファイルが見つかりません: {target_ply}")
        continue

    gt_pcd = o3d.io.read_point_cloud(target_ply)
    gt_pts = np.asarray(gt_pcd.points)
    tree = o3d.geometry.KDTreeFlann(gt_pcd)

    mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
    x_steps = np.arange(mins[0], maxs[0], STRIDE)
    y_steps = np.arange(mins[1], maxs[1], STRIDE)
    z_steps = np.arange(mins[2], maxs[2], STRIDE)
    total_steps = len(x_steps) * len(y_steps) * len(z_steps)

    log_file = f"{OUT_ROOT}/log_{safe_name}.txt"
    processed_keys = set()
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            for line in f: processed_keys.add(line.strip())

    grid_coords = itertools.product(x_steps, y_steps, z_steps)
    file_new_count = 0
    search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

    with open(log_file, "a") as log_f:
        for x, y, z in tqdm(grid_coords, total=total_steps, mininterval=1.0, desc=f"Slicing {safe_name}"):
            key = f"{x:.3f}_{y:.3f}_{z:.3f}"
            if key in processed_keys: continue

            center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2])
            [k, idx, _] = tree.search_radius_vector_3d(center, search_radius)
            if k < MIN_POINTS:
                log_f.write(key + "\n"); continue

            cand = gt_pts[idx]
            mask = np.all((cand >= [x,y,z]) & (cand < [x,y,z] + PATCH_SIZE), axis=1)
            patch = cand[mask]

            if len(patch) < MIN_POINTS:
                log_f.write(key + "\n"); continue

            # Complete作成
            comp_pts = sample_points(patch, PATCH_N)
            comp_pts = comp_pts - np.mean(comp_pts, axis=0)

            # 保存 (IDは global_patch_id を使用)
            file_stem = f"{CATEGORY_ID}-{global_patch_id:06d}"

            patterns = [
                ("00", lambda p: create_random_hole(p)),
                ("01", lambda p: create_viewpoint_scan(p, 'top')),
                ("02", lambda p: create_viewpoint_scan(p, 'front')),
                ("03", lambda p: create_viewpoint_scan(p, 'side')),
                ("04", lambda p: create_viewpoint_scan(p, 'bottom'))
            ]

            partial_subdir = os.path.join(PARTIAL_DIR, file_stem)
            os.makedirs(partial_subdir, exist_ok=True)
            valid_exists = False

            for suffix, func in patterns:
                part_pts = func(comp_pts)
                if len(part_pts) < MIN_POINTS: part_pts = create_random_hole(comp_pts)
                part_pts = sample_points(part_pts, PATCH_N)
                o3d.io.write_point_cloud(os.path.join(partial_subdir, f"{suffix}.pcd"), o3d.geometry.PointCloud(o3d.utility.Vector3dVector(part_pts)), write_ascii=False)
                valid_exists = True

            if valid_exists:
                o3d.io.write_point_cloud(os.path.join(COMPLETE_DIR, f"{file_stem}.pcd"), o3d.geometry.PointCloud(o3d.utility.Vector3dVector(comp_pts)), write_ascii=False)
                global_patch_id += 1 # ★ここでIDを進めるので、ファイルが増えていく
                file_new_count += 1

            log_f.write(key + "\n")

    print(f"   ✅ {safe_name} 追加完了: {file_new_count} 個")

# ==============================================================================
#  📝 リスト更新 (熊本+小倉 全ファイルをまとめてリスト化)
# ==============================================================================
print("📝 Updating split lists...")
# フォルダ内の全データを再取得する
all_names = sorted([d for d in os.listdir(PARTIAL_DIR) if os.path.isdir(os.path.join(PARTIAL_DIR, d))])

if len(all_names) > 0:
    random.shuffle(all_names)
    split = int(len(all_names) * TRAIN_SPLIT)

    with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f:
        f.write("\n".join(all_names[:split]))
    with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f:
        f.write("\n".join(all_names[split:]))

    print(f"✅ 全データ統合リスト更新完了: 合計 {len(all_names)} 件")
else:
    print("⚠️ データがありません")

🎬 追加モードで開始します。
   現在の最終ID: -1
   次の開始ID  : 0 (ここから小倉城を追加します)

🔄 処理中: 熊本正解データ


Slicing 熊本正解データ:   0%|          | 226/57750 [00:00<00:00, 123041.63it/s]


TypeError: can only concatenate list (not "float") to list

完成版手前、ほかの城を入れても続けて学習データセットにはならない

In [ ]:
import os
import sys
import glob
import numpy as np
import random
import itertools
import math
from tqdm import tqdm

try:
    import open3d as o3d
except ImportError:
    print("Open3Dがインストールされていません。インストールします...")
    os.system("pip install open3d")
    import open3d as o3d

# ==============================================================================
#  設定
# ==============================================================================
GT_PLY = "/content/drive/MyDrive/gakusyu/熊本正解データ.ply"
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset/PCN"
CATEGORY_ID = "02691156"

# パッチ設定
PATCH_SIZE = 0.25    # 25cm
STRIDE     = 0.1     # 10cm刻み
PATCH_N    = 2048    # 点数
MIN_POINTS = 50

# 欠損生成の設定
HOLE_MIN_RATIO = 0.25
HOLE_MAX_RATIO = 0.45

# リスト分割
TRAIN_SPLIT = 0.9
RNG_SEED    = 42

# ==============================================================================
#  初期化
# ==============================================================================
if 'google.colab' in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

BASE_DIR = os.path.join(OUT_ROOT, "train")
COMPLETE_DIR = os.path.join(BASE_DIR, "complete", CATEGORY_ID)
PARTIAL_DIR  = os.path.join(BASE_DIR, "partial", CATEGORY_ID)
SPLIT_DIR    = os.path.join(OUT_ROOT, "splits")

os.makedirs(COMPLETE_DIR, exist_ok=True)
os.makedirs(PARTIAL_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

LOG_FILE = f"{OUT_ROOT}/processed_log_multi_v2_{CATEGORY_ID}.txt"

# ==============================================================================
#  関数: 様々な欠損を作る
# ==============================================================================

def sample_points(pts, n_points):
    """ 点数を揃える """
    if len(pts) == 0: return pts
    if len(pts) >= n_points:
        return pts[np.random.choice(len(pts), n_points, replace=False)]
    else:
        return pts[np.random.choice(len(pts), n_points, replace=True)]

def create_random_hole(pts):
    """ 00: ランダムな球状の穴 """
    mins, maxs = pts.min(axis=0), pts.max(axis=0)
    diag = np.linalg.norm(maxs - mins)
    center = pts[np.random.randint(0, len(pts))]
    radius = diag * np.random.uniform(HOLE_MIN_RATIO, HOLE_MAX_RATIO)

    dists = np.linalg.norm(pts - center, axis=1)
    return pts[dists > radius] # 穴の外側を残す

def create_viewpoint_scan(pts, view_axis):
    """
    指定した方向からのスキャンをシミュレート
    view_axis: 'top'(+Y), 'bottom'(-Y), 'front'(+Z), 'side'(+X)
    """
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)

    # カメラ位置をパッチサイズの100倍遠くに設定
    diameter = np.linalg.norm(pts.max(axis=0) - pts.min(axis=0))
    camera_dist = diameter * 100

    if view_axis == 'top':
        camera = [0, camera_dist, 0]     # 真上から
    elif view_axis == 'bottom':
        camera = [0, -camera_dist, 0]    # ★真下から（屋根が見えなくなる）
    elif view_axis == 'front':
        camera = [0, 0, camera_dist]     # 正面から
    elif view_axis == 'side':
        camera = [camera_dist, 0, 0]     # 右側面から
    else:
        camera = [0, 0, camera_dist]

    # 隠点除去 (Hidden Point Removal)
    # カメラから見える点だけを残す
    _, pt_map = pcd.hidden_point_removal(camera, radius=camera_dist * 1000)

    visible_pts = pts[pt_map]
    return visible_pts

# ==============================================================================
#  メイン処理
# ==============================================================================
print(f"🔄 Loading GT: {GT_PLY} ...")
if not os.path.exists(GT_PLY):
    sys.exit(1)

gt_pcd = o3d.io.read_point_cloud(GT_PLY)
gt_pts = np.asarray(gt_pcd.points)

print("🌳 Building KDTree...")
tree = o3d.geometry.KDTreeFlann(gt_pcd)

mins, maxs = gt_pts.min(axis=0), gt_pts.max(axis=0)
x_steps = np.arange(mins[0], maxs[0], STRIDE)
y_steps = np.arange(mins[1], maxs[1], STRIDE)
z_steps = np.arange(mins[2], maxs[2], STRIDE)
total_steps = len(x_steps) * len(y_steps) * len(z_steps)

# 開始ID決定
existing = glob.glob(os.path.join(COMPLETE_DIR, "*.pcd"))
patch_id = 0
if existing:
    try:
        patch_id = max([int(os.path.basename(f).split('-')[-1].replace('.pcd','')) for f in existing]) + 1
    except: pass
print(f"▶ Start ID: {patch_id:06d}")

# ログ読み込み
processed_keys = set()
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, "r") as f:
        for line in f: processed_keys.add(line.strip())

# ループ
print("🚀 Generating 5-View Pairs (Hole, Top, Front, Side, Bottom)...")
grid_coords = itertools.product(x_steps, y_steps, z_steps)
new_count = 0
search_radius = (PATCH_SIZE * math.sqrt(3) / 2.0) * 1.05

with open(LOG_FILE, "a") as log_f:
    for x, y, z in tqdm(grid_coords, total=total_steps, mininterval=1.0, desc="Processing"):

        key = f"{x:.3f}_{y:.3f}_{z:.3f}"
        if key in processed_keys: continue

        # 1. Complete 切り出し
        center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2])
        [k, idx, _] = tree.search_radius_vector_3d(center, search_radius)
        if k < MIN_POINTS:
            log_f.write(key + "\n"); continue

        cand = gt_pts[idx]
        mask = np.all((cand >= [x,y,z]) & (cand < [x,y,z] + PATCH_SIZE), axis=1)
        patch = cand[mask]

        if len(patch) < MIN_POINTS:
            log_f.write(key + "\n"); continue

        # 2. 正規化 (Complete)
        comp_pts = sample_points(patch, PATCH_N)
        center_pts = np.mean(comp_pts, axis=0)
        comp_pts = comp_pts - center_pts

        # 3. 5種類の Partial を生成して保存
        file_stem = f"{CATEGORY_ID}-{patch_id:06d}"

        # Complete保存
        o3d.io.write_point_cloud(
            os.path.join(COMPLETE_DIR, f"{file_stem}.pcd"),
            o3d.geometry.PointCloud(o3d.utility.Vector3dVector(comp_pts)),
            write_ascii=False
        )

        partial_subdir = os.path.join(PARTIAL_DIR, file_stem)
        os.makedirs(partial_subdir, exist_ok=True)

        # パターン定義 (ここを強化！)
        patterns = [
            ("00", lambda p: create_random_hole(p)),           # ランダム穴
            ("01", lambda p: create_viewpoint_scan(p, 'top')),    # 上から (下なし)
            ("02", lambda p: create_viewpoint_scan(p, 'front')),  # 前から (後ろなし)
            ("03", lambda p: create_viewpoint_scan(p, 'side')),   # 横から (逆なし)
            ("04", lambda p: create_viewpoint_scan(p, 'bottom'))  # ★下から (上なし)
        ]

        valid_partial_exists = False

        for suffix, func in patterns:
            part_pts = func(comp_pts)

            if len(part_pts) < MIN_POINTS:
                # 視点によっては何も見えないことがあるため、その場合はランダム穴で代用
                part_pts = create_random_hole(comp_pts)

            part_pts = sample_points(part_pts, PATCH_N)

            o3d.io.write_point_cloud(
                os.path.join(partial_subdir, f"{suffix}.pcd"),
                o3d.geometry.PointCloud(o3d.utility.Vector3dVector(part_pts)),
                write_ascii=False
            )
            valid_partial_exists = True

        if valid_partial_exists:
            patch_id += 1
            new_count += 1

        log_f.write(key + "\n")

print(f"\n✨ 生成完了: {new_count} セット (Total ID: {patch_id})")

print("📝 Updating lists...")
all_names = sorted([d for d in os.listdir(PARTIAL_DIR) if os.path.isdir(os.path.join(PARTIAL_DIR, d))])
if all_names:
    random.shuffle(all_names)
    split = int(len(all_names) * TRAIN_SPLIT)
    with open(os.path.join(SPLIT_DIR, "train.txt"), "w") as f: f.write("\n".join(all_names[:split]))
    with open(os.path.join(SPLIT_DIR, "test.txt"), "w") as f: f.write("\n".join(all_names[split:]))
    print("✅ Done!")

completeもpartialもどちらも直接pcd形式で出してる

In [ ]:
# ==============================================================================
# SeedFormer / PCN 対応 データセット生成スクリプト
# ==============================================================================
# 機能:
# 1. 「完全点群(GT)」と「欠損点群(Hole)」の両方を読み込む
# 2. 同じ位置でスライスし、pair データ (partial / complete) を作成する
# 3. GTの重心を基準に正規化を行い、位置ズレを防ぐ
# 4. train/test リストを作成する
# 5. 途中再開(レジューム)機能付き
# ==============================================================================

import os
import sys
import glob
import numpy as np
import random
import itertools
import shutil
from tqdm import tqdm

# Google Colabで実行する場合、Open3Dが必要です
try:
    import open3d as o3d
except ImportError:
    print("Open3Dがインストールされていません。インストールします...")
    os.system("pip install open3d")
    import open3d as o3d

# ================================================
#  設定（ここを自分の環境に合わせて変更）
# ================================================

# 入力ファイル
GT_PLY   = "/content/drive/MyDrive/gakusyu/熊本正解データ.ply"  # 完全データ
HOLE_PLY = "/content/drive/MyDrive/gakusyu/kesson2.ply"     # 欠損データ

# 保存先ルートディレクトリ (★ここを変更しました)
# Google Driveの「マイドライブ」直下に、分かりやすい名前で保存します
# 好きな名前に書き換えてOKです
OUT_ROOT = "/content/drive/MyDrive/My_PCN_Dataset"

# カテゴリID（ShapeNet形式）
# 実際の保存先は .../My_PCN_Dataset/shapenet_pc/02691156/... のようになります
CATEGORY_ID = "02691156"

# パッチ作成設定
PATCH_SIZE = 0.25    # 1辺の長さ (単位: m)
STRIDE     = 0.1     # ずらし幅 (単位: m)
PATCH_N    = 2048    # 1パッチあたりの点数
MIN_POINTS = 50      # この点数未満はゴミとして捨てる

# 学習データの分割割合
TRAIN_SPLIT = 0.9    # 90% Train, 10% Test
RNG_SEED    = 42

# ================================================
#  初期化処理
# ================================================

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

# ディレクトリ作成 (PCN形式: partial/complete)
BASE_DIR = f"{OUT_ROOT}/shapenet_pc/{CATEGORY_ID}"
PARTIAL_DIR  = os.path.join(BASE_DIR, "partial")
COMPLETE_DIR = os.path.join(BASE_DIR, "complete")
LIST_DIR     = f"{OUT_ROOT}/ShapeNet55_splits/Main_Split"

os.makedirs(PARTIAL_DIR, exist_ok=True)
os.makedirs(COMPLETE_DIR, exist_ok=True)
os.makedirs(LIST_DIR, exist_ok=True)

# ログファイル
LOG_FILE = f"{OUT_ROOT}/processed_log_{CATEGORY_ID}.txt"

# ================================================
#  1. 点群読み込み & KDTree構築
# ================================================
def load_pcd(path):
    print(f"Loading {path} ...")
    if not os.path.exists(path):
        print(f"❌ エラー: ファイルが見つかりません -> {path}")
        sys.exit(1)
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points)
    print(f"  -> Points: {len(pts)}")
    return pcd, pts

# 完全データ(GT)と欠損データ(Hole)の両方を読み込み
gt_pcd, gt_pts = load_pcd(GT_PLY)
hole_pcd, hole_pts = load_pcd(HOLE_PLY)

# 探索範囲はGTを基準にする
global_min = gt_pts.min(axis=0)
global_max = gt_pts.max(axis=0)

print("Building KDTree for acceleration...")
# 両方のKDTreeを作成（高速切り出しのため）
tree_gt = o3d.geometry.KDTreeFlann(gt_pcd)
tree_hole = o3d.geometry.KDTreeFlann(hole_pcd)

# グリッド設定
margin = 0.05
x_steps = np.arange(global_min[0]-margin, global_max[0]+margin, STRIDE)
y_steps = np.arange(global_min[1]-margin, global_max[1]+margin, STRIDE)
z_steps = np.arange(global_min[2]-margin, global_max[2]+margin, STRIDE)

total_steps = len(x_steps) * len(y_steps) * len(z_steps)
print(f"Grid setup: Size={PATCH_SIZE}m, Stride={STRIDE}m")
print(f"  -> Total search grids: {total_steps}")

# ================================================
#  2. レジューム（再開）情報
# ================================================
processed_keys = set()
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, "r") as f:
        for line in f:
            processed_keys.add(line.strip())
    print(f"🔄 Resuming... skipped {len(processed_keys)} grids.")

# 既存ファイルからID決定 (completeフォルダを基準にする)
existing_files = glob.glob(os.path.join(COMPLETE_DIR, "*.pcd"))
current_max_id = 0
if len(existing_files) > 0:
    try:
        ids = [int(os.path.basename(f).split('-')[-1].replace('.pcd','')) for f in existing_files]
        current_max_id = max(ids) + 1
    except:
        current_max_id = len(existing_files)

patch_id = current_max_id
print(f"Start generating from ID: {patch_id}")

# ================================================
#  3. パッチ生成メインループ
# ================================================
print("🚀 Start generating PAIRS (partial & complete)...")
grid_coords = itertools.product(x_steps, y_steps, z_steps)
new_count = 0
search_radius = (PATCH_SIZE * np.sqrt(3) / 2.0) * 1.01

def get_points_in_box(tree, all_pts, center, radius, min_b, max_b):
    # 半径探索
    [k, idx, _] = tree.search_radius_vector_3d(center, radius)
    if k < MIN_POINTS:
        return np.empty((0, 3))

    # ボックス判定
    cand = all_pts[idx]
    mask = np.all((cand >= min_b) & (cand < max_b), axis=1)
    return cand[mask]

def sample_points(pts, n_points):
    if len(pts) == 0: return pts
    if len(pts) >= n_points:
        return pts[np.random.choice(len(pts), n_points, replace=False)]
    else:
        return pts[np.random.choice(len(pts), n_points, replace=True)]

with open(LOG_FILE, "a") as log_f:
    for x, y, z in tqdm(grid_coords, total=total_steps, mininterval=1.0, desc="Slicing"):

        key = f"{x:.3f}_{y:.3f}_{z:.3f}"
        if key in processed_keys:
            continue

        center = np.array([x + PATCH_SIZE/2, y + PATCH_SIZE/2, z + PATCH_SIZE/2])
        min_bound = np.array([x, y, z])
        max_bound = min_bound + PATCH_SIZE

        # 1. Complete (GT) の切り出し
        gt_patch = get_points_in_box(tree_gt, gt_pts, center, search_radius, min_bound, max_bound)

        # GTが少なすぎる場合は学習データにならないのでスキップ
        if len(gt_patch) < MIN_POINTS:
            log_f.write(key + "\n")
            continue

        # 2. Partial (Hole) の切り出し
        hole_patch = get_points_in_box(tree_hole, hole_pts, center, search_radius, min_bound, max_bound)

        # Partial側が空っぽ(0点)でも、補完タスクとしては「全欠損」としてあり得るが、
        # 学習が難しくなるため、ある程度点があるものだけ採用するか、
        # ユーザーの要件次第。ここでは「Partialも少しは点がないと位置合わせできない」と判断しスキップ。
        if len(hole_patch) < MIN_POINTS:
            log_f.write(key + "\n")
            continue

        # 3. サンプリング
        gt_final = sample_points(gt_patch, PATCH_N)
        hole_final = sample_points(hole_patch, PATCH_N)

        # 4. ★正規化 (位置合わせ)
        # 重要: 「GTの中心」を使って両方をシフトする。
        # Partial自身の中心を使うと、PartialとCompleteの位置関係がズレてしまう。
        patch_center = np.mean(gt_final, axis=0)

        gt_final   = gt_final - patch_center
        hole_final = hole_final - patch_center

        # 5. 保存
        file_name = f"{CATEGORY_ID}-{patch_id:06d}.pcd"

        # Complete保存
        o3d.io.write_point_cloud(
            os.path.join(COMPLETE_DIR, file_name),
            o3d.geometry.PointCloud(o3d.utility.Vector3dVector(gt_final)),
            write_ascii=False
        )
        # Partial保存
        o3d.io.write_point_cloud(
            os.path.join(PARTIAL_DIR, file_name),
            o3d.geometry.PointCloud(o3d.utility.Vector3dVector(hole_final)),
            write_ascii=False
        )

        patch_id += 1
        new_count += 1
        log_f.write(key + "\n")

print(f"\n✨ 生成完了: {new_count} ペア (Total ID: {patch_id})")

# ================================================
#  4. リスト作成
# ================================================
print("\n📝 Generating split lists...")
all_files = sorted([os.path.basename(f) for f in glob.glob(os.path.join(COMPLETE_DIR, "*.pcd"))])

if len(all_files) > 0:
    random.shuffle(all_files)
    split_idx = int(len(all_files) * TRAIN_SPLIT)
    train_files = all_files[:split_idx]
    test_files  = all_files[split_idx:]

    with open(os.path.join(LIST_DIR, "train.txt"), "w") as f:
        f.write("\n".join(train_files))
    with open(os.path.join(LIST_DIR, "test.txt"), "w") as f:
        f.write("\n".join(test_files))

    print(f"✅ リスト作成完了: Train={len(train_files)}, Test={len(test_files)}")
else:
    print("⚠️ ファイルなし")

print("\n🎉 完了！")

Loading /content/drive/MyDrive/gakusyu/熊本正解データ.ply ...
  -> Points: 970651
Loading /content/drive/MyDrive/gakusyu/kesson2.ply ...
  -> Points: 468677
Building KDTree for acceleration...
Grid setup: Size=0.25m, Stride=0.1m
  -> Total search grids: 62424
Start generating from ID: 0
🚀 Start generating PAIRS (partial & complete)...


Slicing: 100%|██████████| 62424/62424 [01:30<00:00, 686.95it/s] 



✨ 生成完了: 6124 ペア (Total ID: 6124)

📝 Generating split lists...
✅ リスト作成完了: Train=5511, Test=613

🎉 完了！


In [ ]:
import glob, os, shutil
from tqdm import tqdm  # プログレスバー用のライブラリ

# ============
# 1. パス設定
# ============
SRC_ROOT = '/content/drive/MyDrive/My_PCN_Dataset1/shapenet_pc/02691156'
DST_ROOT = '/content/My_PCN_Dataset/PCN/train'

# ==================
# 2. partial の一覧
# ==================
partial_files = sorted(glob.glob(os.path.join(SRC_ROOT, 'partial', '*.pcd')))
total = len(partial_files)

print(f"🔍 partial 点群の総数: {total}")

# ======================
# 3. PCN 形式へコピー
# ======================
processed = 0
skipped = 0

print("🚀 処理を開始します...")

# ★修正点: tqdmでラップして、プログレスバーを表示
for p_path in tqdm(partial_files, desc="Copying"):
    fname = os.path.basename(p_path)         # 例: 02691156-001437.pcd
    stem  = os.path.splitext(fname)[0]       # 例: 02691156-001437

    # 完成形（complete）を探す
    c_path = os.path.join(SRC_ROOT, 'complete', fname)

    # ★修正点: 毎回の「処理中」printを削除しました

    if not os.path.exists(c_path):
        # ★修正点: スキップ時のprintも削除（数が多いと邪魔なため）
        skipped += 1
        continue

    # ---------- partial 側 ----------
    dst_partial_dir = os.path.join(DST_ROOT, 'partial', 'airplane', stem)
    os.makedirs(dst_partial_dir, exist_ok=True)
    shutil.copy(p_path, os.path.join(dst_partial_dir, '00.pcd'))

    # ---------- complete 側 ----------
    dst_complete_dir = os.path.join(DST_ROOT, 'complete', 'airplane')
    os.makedirs(dst_complete_dir, exist_ok=True)
    shutil.copy(c_path, os.path.join(dst_complete_dir, f"{stem}.pcd"))

    processed += 1
    # ★修正点: 完了時のprintを削除しました

# ======================
# 4. 結果レポート
# ======================
print("\n=======================")
print("  ⭐ 変換処理 完了 ⭐")
print("=======================\n")
print(f"✔ 成功: {processed} 件")
print(f"⚠ スキップ: {skipped} 件")
print(f"📁 出力先: {DST_ROOT}")

プログレスバーにした。

環境構築ここからランタイムの切り替え

